[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/two-phaseflow/JED116F/blob/main/notebooks/02_well_tests_pressure_transients.ipynb)

Click the badge to open this notebook in Google Colab. In Colab, choose **File > Save a copy in Drive** before you start, so your work is kept.

# JEÐ116F Notebook 02. Well tests: injectivity, the pressure diffusion equation, and transmissivity from a pressure record

Lecture 6, Thursday 22 October 2026. Parts 1 to 4 are started in class, in three blocks between the board sessions. The completed notebook is Assignment 3; the last section says what to hand in.

**Cell types**

- **A. Run and look.** The code is complete. Run it and read the output.
- **B. Fill the gaps.** Replace every `___` with a number or a short expression. For the K-37 cells done in class the expected values are given in the text; for the other wells they are not, since those are the assignment.
- **Possible extensions.** Open questions for when you finish early.

**Rules for this course**

1. Keep **units** next to every number in your code comments.
2. You may use AI tools for error messages, plotting and coding ideas. You must be able to explain every line you hand in.

**Parts**

1. The injectivity index from three real step tests (K-37, K-38, NJ-15)
2. The pressure diffusion equation in code: forecasting the pressure funnel
3. Transmissivity and kh from the straight line (K-37 in code, then K-38 and NJ-15)
4. Interference: storativity from a second well (Ohaaki BR19 to BR23)
5. A discharge test and the James lip-pressure method (Ohaaki BR20)

**Assignment 3** at the end: what to hand in.

**Symbols used throughout** (mass form, as in the lecture)

| symbol | meaning | unit |
|---|---|---|
| $W$ | mass flow rate into the well (injection positive) | kg/s |
| $P$ | pressure at the feed zone | Pa (we print bar) |
| $\nu = \mu/\rho$ | kinematic viscosity of the reservoir fluid | m²/s |
| $T = kh/\nu$ | transmissivity | m s |
| $kh$ | permeability times thickness | m³ (1 darcy-metre = 1e-12 m³) |
| $S = \phi c_t h$ | storativity, as the slides and reports quote it | m/Pa |
| $S_m = \rho S$ | mass storativity, used with $W$ in kg/s | kg/(m² Pa) |
| $\rho$ | density of the reservoir fluid | kg/m³ |
| $c_t$ | total compressibility, fluid plus pore space | 1/Pa |
| $r_\mathrm{well}$ | well radius | m |
| $s$ | skin factor | dimensionless |
| $E_1$ | the exponential integral, `scipy.special.exp1` | |

## Part 0. Setup

The data files are in the course repository on GitHub. The cell below downloads the ones this notebook needs. Each CSV file starts with comment lines (beginning with `#`) that give the source of the data and the test details. `pandas` skips these lines with `comment='#'`. Water properties come from the IAPWS-IF97 formulation in the `iapws` package; no fluid property is typed in by hand.

In [ ]:
%pip install -q iapws

import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import exp1          # the exponential integral E1
from scipy.integrate import solve_ivp   # an ODE solver, used in Part 2
from scipy.optimize import curve_fit    # least-squares fitting, used in Part 4
from iapws import IAPWS97

plt.rcParams.update({"font.size": 13, "figure.figsize": (8, 5), "axes.grid": True})

BASE = "https://raw.githubusercontent.com/two-phaseflow/JED116F/main/data/"
FILES = ["K37_steptest_2008-01-18_raw.csv", "K38_steptest_2008-07-20_record.csv", "NJ15_steptest_1995-10-21_record.csv",
         "BR23_interference_1980.csv", "BR20_output_test.csv"]
DATA = os.environ.get("JED116F_DATA", "data")      # local folder if it exists, otherwise download
os.makedirs(DATA, exist_ok=True)
for f in FILES:
    if not os.path.exists(os.path.join(DATA, f)):
        urllib.request.urlretrieve(BASE + f, os.path.join(DATA, f))

def load(name):
    '''Read one course CSV, skipping the comment header.'''
    return pd.read_csv(os.path.join(DATA, name), comment="#")

def header(name, n=4):
    '''Print the first n comment lines of a course CSV: the data source and test details.'''
    with open(os.path.join(DATA, name)) as f:
        for line in f:
            if not line.startswith("#"): break
            print(line.rstrip()[:160])

def nu_water(T_C, P_MPa=10.0):
    '''Kinematic viscosity of liquid water, m2/s, from IAPWS-IF97.'''
    w = IAPWS97(P=P_MPa, T=T_C + 273.15)
    return w.mu / w.rho

def rho_water(T_C, P_MPa=10.0):
    '''Density of liquid water, kg/m3, from IAPWS-IF97.'''
    return IAPWS97(P=P_MPa, T=T_C + 273.15).rho

def c_water(P_MPa, dP_MPa=0.5):
    '''Isothermal compressibility of saturated liquid water at P_MPa, 1/Pa: (1/rho) drho/dP by finite difference of IAPWS densities.'''
    T = IAPWS97(P=P_MPa, x=0).T
    r1, r2 = IAPWS97(P=P_MPa + dP_MPa, T=T).rho, IAPWS97(P=P_MPa + 2 * dP_MPa, T=T).rho
    return (r2 - r1) / (dP_MPa * 1e6) / r1

def sat_liquid(P_MPa):
    '''Saturated liquid water at pressure P_MPa: boiling temperature (C), density (kg/m3), kinematic viscosity (m2/s).'''
    w = IAPWS97(P=P_MPa, x=0)
    return w.T - 273.15, w.rho, w.mu / w.rho

g = 9.81                                   # m/s2
DARCY_M = 1e-12                            # m3, one darcy-metre
BAR = 1e5                                  # Pa
bar_per_m_cold = rho_water(20, 0.1) * g / BAR   # pressure change for 1 m of cold water column near the surface, bar

print("cold water, 20 C:  nu = %.2e m2/s, rho = %.0f kg/m3, 1 m of column = %.3f bar" % (nu_water(20), rho_water(20, 0.1), bar_per_m_cold))
print("hot water, 300 C, 100 bar:  nu = %.2e m2/s, rho = %.0f kg/m3" % (nu_water(300), rho_water(300)))

## Part 1. The injectivity index

The injectivity index is the first number read from a step test:

$$ II = \frac{|\Delta W|}{|\Delta p|} $$

$\Delta W$ is the change in injection rate (kg/s; for cold water 1 L/s is 1 kg/s) and $\Delta p$ the change in the settled pressure at the feed zone (bar). Units: (kg/s) per bar.

### A. The K-37 record

Krafla well K-37, 18 January 2008, the last day of drilling. A gauge at 1420 m, the main feed zone, logged the pressure every 5 seconds while the pump rate was changed three times. This is the raw data obtained from https://uos.is/boreholes/58037.

In [ ]:
header("K37_steptest_2008-01-18_raw.csv")
k37 = load("K37_steptest_2008-01-18_raw.csv")
k37["t_h"] = k37["t_s"] / 3600            # hours since the first rate change at 14:17
display(k37.head())

def plot_record(df, t_col, title):
    '''Pressure (left axis) and injection rate (right axis) against time for one step test.'''
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df[t_col], df.p_barg, ".", ms=3, color="#2a78d6")
    ax.set_xlabel("time, hours"); ax.set_ylabel("pressure at the feed zone, bar-g", color="#2a78d6")
    ax2 = ax.twinx(); ax2.step(df[t_col], df.rate_L_s, where="post", color="#1baf7a")
    ax2.set_ylabel("injection rate, L/s", color="#1baf7a"); ax2.grid(False)
    ax.set_title(title, loc="left")

plot_record(k37, "t_h", "K-37, 18 January 2008: three rate changes, three hours each")

### B1.1 The index from each step

The pressure before the test, at 20 L/s, was 80.0 bar-g. For each step the code below reads the settled pressure as the mean of the last 10 minutes of that step and gives you the rate change `dW` and the pressure change `dp`. Fill in the equation for the injectivity index.

In [ ]:
p_before = 80.0                                   # bar-g, at 20 L/s before the test
rates = {0: 20.0, 1: 35.0, 2: 50.0, 3: 20.0}      # L/s before the test and after each change

II = {}
p_end = {0: p_before}
for step in (1, 2, 3):
    d = k37[k37.step == step]
    last10 = d[d.t_s >= d.t_s.max() - 600]        # the last 10 minutes of the step
    p_end[step] = last10.p_barg.mean()            # settled pressure at the end of this step, bar
    dW = rates[step] - rates[step - 1]            # change in rate, kg/s (1 L/s of cold water is 1 kg/s)
    dp = p_end[step] - p_end[step - 1]            # change in settled pressure, bar
    II[step] = ___   # the injectivity index, (kg/s) per bar
    print(f"step {step}: {rates[step-1]:.0f} to {rates[step]:.0f} L/s, pressure {p_end[step-1]:.1f} to {p_end[step]:.1f} bar, II = {II[step]:.2f} (kg/s)/bar")

### A. Does it matter when you read the settled pressure?

The pressure keeps changing slowly during a step. Read the pressure 15 minutes, one hour and three hours after the rate change and see how much the index changes.

In [ ]:
d = k37[k37.step == 1]
t0 = d.t_s.min()
for minutes in (15, 60, 180):
    at = d[(d.t_s >= t0 + minutes * 60 - 150) & (d.t_s <= t0 + minutes * 60 + 150)].p_barg.mean()   # 5 min window
    print(f"read after {minutes:3d} min: pressure {at:.2f} bar, II = {15 / (at - 80.0):.2f} (kg/s)/bar")
print("The index read early is too high because the pressure is still rising. The standard practice is to read it at the end of a step of several hours.")

### A. Two more step tests: K-38 and NJ-15

K-38 (Krafla, 20 July 2008) and NJ-15 (Nesjavellir, 21 October 1995). These records were digitized from the figures in Ntihabose (2014), so the points are about two minutes apart and the pressures are good to about 0.1 bar. Both files have the columns `t_h` (hours since midnight), `p_barg` and `rate_L_s`. Look at the two records before you compute anything.

In [ ]:
header("K38_steptest_2008-07-20_record.csv", 2)
k38_rec = load("K38_steptest_2008-07-20_record.csv")
plot_record(k38_rec, "t_h", "K-38, 20 July 2008")

header("NJ15_steptest_1995-10-21_record.csv", 2)
nj15_rec = load("NJ15_steptest_1995-10-21_record.csv")
plot_record(nj15_rec, "t_h", "NJ-15, 21 October 1995")

### B1.2 The index for every step of K-38 and NJ-15

The function `step_pressures` finds every rate change in a record and returns, for each one, the rate before and after and the settled pressure before and after (median of the last 30 minutes of each step). Fill in the equation for the index.

In [ ]:
def settled(df, t_end_h):
    '''Settled pressure before time t_end_h: median of the points in the last 30 minutes, leaving out the last 2 minutes
    (a digitized point there can already belong to the next step).'''
    before = df[df.t_h <= t_end_h - 2 / 60]
    win = before[before.t_h >= t_end_h - 0.5]
    if len(win) < 3:                     # digitized records have gaps; then take the last three points of the step
        win = before.tail(3)
    return win.p_barg.median()

def step_pressures(df):
    '''One row per rate change: time, rate before and after (L/s), settled pressure before and after (bar-g).'''
    changes = list(df.index[df.rate_L_s.diff().fillna(0) != 0])   # row numbers where the rate changes
    ends = [df.loc[j, "t_h"] for j in changes[1:]] + [df.t_h.max() + 2 / 60]   # end of each step
    rows = []
    for i, t_end in zip(changes, ends):
        t_change = df.loc[i, "t_h"]
        rows.append({"t_h": t_change, "rate_from": df.loc[i - 1, "rate_L_s"], "rate_to": df.loc[i, "rate_L_s"],
                     "p_before": settled(df, t_change), "p_after": settled(df, t_end)})
    return pd.DataFrame(rows)

steps_all = {}
for name, rec in (("K-38", k38_rec), ("NJ-15", nj15_rec)):
    steps = step_pressures(rec)
    dW = steps.rate_to - steps.rate_from          # change in rate, kg/s
    dp = steps.p_after - steps.p_before           # change in settled pressure, bar
    steps["II"] = ___   # the injectivity index, (kg/s) per bar
    steps_all[name] = steps
    print(name); display(steps.round(2))

Compare the three wells. Icelandic high-temperature wells mostly have indices between 1 and 10 (kg/s) per bar; below 2 is a poor well, above 10 a good one.

## Part 2. The pressure diffusion equation in code

In the lecture we took one ring of the reservoir, between $r$ and $r + dr$, and wrote two equations for it.

Darcy's law gives the flow across each face of the ring (kg/s, counted toward the well):

$$ w(r) = 2\pi r T\,\frac{\partial P}{\partial r} $$

The mass balance for the ring: flow in across the outer face minus flow out across the inner face equals the water the ring releases per second:

$$ w(r + dr) - w(r) = S_m\, 2\pi r\, dr\, \frac{\partial P}{\partial t} $$

Put Darcy in for $w$ and divide by $2\pi r\,dr$, and the two lines become the pressure diffusion equation:

$$ \frac{T}{r}\frac{\partial}{\partial r}\left(r\frac{\partial P}{\partial r}\right) = S_m\frac{\partial P}{\partial t}
\qquad\text{or, expanded,}\qquad
T\left(\frac{\partial^2 P}{\partial r^2} + \frac{1}{r}\frac{\partial P}{\partial r}\right) = S_m\frac{\partial P}{\partial t} $$

$S_m = \rho\,\phi c_t h$ is the mass storativity, kg/(m² Pa). It is the storativity $S = \phi c_t h$ (m/Pa) times the fluid density. The mass form is the one that belongs with $W$ in kg/s and $T = kh/\nu$ in m s. $T/S_m$ is the hydraulic diffusivity, m²/s.

Theis solved this equation analytically for one case: one well at a constant rate in a layer of unlimited extent. A numerical model does not need that case. It needs the two equations above written for a finite number of rings, and it advances the pressure in every ring through many small time steps. A reservoir simulator does the same thing in two or three dimensions with more physics added. The model below is about 20 lines of code.

### A. From the equation to a function

Step 1. Divide the reservoir into $N$ rings, from the well wall out to a far boundary. The pressure changes fastest near the well, so the rings are thin there and wide far out.

Step 2. Darcy on each face: `w = 2 pi r_face T (P_outer_ring - P_inner_ring) / distance`. This is the first slide equation, with the derivative replaced by a difference between the two rings that share the face.

Step 3. Mass balance for each ring, solved for the rate of change: `dP/dt = (w_outer_face - w_inner_face) / (S_m 2 pi r dr)`. This is the second slide equation.

Step 4. Put steps 2 and 3 in a Python function `dPdt(t, P)` that returns the rate of change of pressure in every ring. This function is the diffusion equation in discrete form. Pass it to an ODE solver, `scipy.integrate.solve_ivp`, which chooses and takes the time steps.

The two ends of the grid need a rule of their own. At the well wall the flow across the face is the well rate, $w = -W$ for injection (water goes outward, away from the well). At the far boundary the pressure stays at $P_\mathrm{res}$.

In [ ]:
def make_model(T, S_m, W, r_well=0.1, r_outer=20000.0, N=250, P_res=0.0):
    '''Radial pressure diffusion around one well, ring by ring.
    T transmissivity (m s), S_m mass storativity (kg/(m2 Pa)), W rate injected into the well (kg/s, injection positive).
    Returns the ring centres r (m) and the function dPdt(t, P) that solve_ivp needs. P is the pressure change, Pa.'''
    r_face = np.geomspace(r_well, r_outer, N + 1)      # N + 1 faces, thin rings near the well
    r = np.sqrt(r_face[:-1] * r_face[1:])              # ring centres
    dr = np.diff(r_face)                               # ring widths
    two_pi_r_dr = 2 * np.pi * r * dr                   # plan-view area of each ring, m2

    def dPdt(t, P):
        # Step 2. Darcy on each face, flow counted toward the well:  w = 2 pi r T dP/dr
        w = np.empty(N + 1)
        w[1:-1] = 2 * np.pi * r_face[1:-1] * T * (P[1:] - P[:-1]) / (r[1:] - r[:-1])
        w[0] = -W                                                            # well wall: the well rate leaves outward
        w[-1] = 2 * np.pi * r_face[-1] * T * (P_res - P[-1]) / (r_face[-1] - r[-1])   # far boundary held at P_res
        # Step 3. Mass balance for each ring:  w(r + dr) - w(r) = S_m 2 pi r dr dP/dt
        return (w[1:] - w[:-1]) / (S_m * two_pi_r_dr)

    return r, dPdt

# K-37 inputs. The feed zone is at 1420 m and the pressure there is the 80 bar-g read in Part 1. The Krafla reservoir is at
# the boiling point at this depth, so the reservoir liquid is saturated water at the feed-zone pressure.
P_feed_MPa = (p_before + 1.0) / 10             # MPa, absolute pressure at the feed zone
T_sat_res, rho_res, nu_res = sat_liquid(P_feed_MPa)   # saturated liquid water at the feed-zone pressure
T_k37 = 1.6e-5                                 # m s, transmissivity from the slope of step 3 (lecture slide; Part 3 recomputes it)
# Storativity from its definition S_m = rho phi c_t h. Nothing is measured here, so every input is an assumption, written down:
phi = 0.10                                     # porosity of the fractured feed zone, assumed
h_feed = 150.0                                 # m, the main feed zone 1300 to 1450 m (temperature logs while injecting)
c_pore = 5e-10                                 # 1/Pa, pore compressibility of fractured basalt, assumed
c_w = c_water(P_feed_MPa)                      # 1/Pa, compressibility of the hot liquid water, from IAPWS (Part 0)
c_t = c_w + c_pore                             # 1/Pa, total compressibility
S_k37 = phi * c_t * h_feed                     # m/Pa, volume form
Sm_k37 = rho_res * S_k37                       # kg/(m2 Pa), mass form
W_k37 = 30.0                                   # kg/s, the rate change of step 3 (the sign only flips the curves)
print("feed-zone pressure %.1f MPa, boiling point there %.0f C, liquid density %.0f kg/m3" % (P_feed_MPa, T_sat_res, rho_res))

r, dPdt = make_model(T_k37, Sm_k37, W_k37)
print("grid:", len(r), "rings from %.2f m to %.0f m; first ring %.3f m wide, last %.0f m" % (r[0], r[-1], r[1] - r[0], r[-1] - r[-2]))
print("c_w = %.1e 1/Pa, S = %.1e m/Pa, S_m = %.1e kg/(m2 Pa), diffusivity T/S_m = %.2f m2/s" % (c_w, S_k37, Sm_k37, T_k37 / Sm_k37))

### A. Forecast the funnel

Start with no pressure change anywhere, start the well at $t = 0$, and ask the solver for the pressure at six times. The thin rings near the well make the equations stiff; `BDF` is an implicit method made for stiff problems.

In [ ]:
times_h = np.array([0.01, 0.1, 1, 10, 100, 1000])          # hours
sol = solve_ivp(dPdt, (0, times_h[-1] * 3600), np.zeros(len(r)), t_eval=times_h * 3600, method="BDF", rtol=1e-6, atol=1.0)
print("solver status:", sol.message)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
for j, th in enumerate(times_h):
    lab = f"{th*60:.0f} min" if th < 1 else f"{th:.0f} h"
    a1.plot(r, sol.y[:, j] / BAR, label=lab); a2.plot(r, sol.y[:, j] / BAR, label=lab)
a1.set_xlim(0, 1000); a1.set_xlabel("distance from the well, m"); a1.set_ylabel("pressure change, bar"); a1.set_title("linear distance", loc="left")
a2.set_xscale("log"); a2.set_xlabel("distance from the well, m (log)"); a2.set_title("log distance: the funnel is a straight line near the well", loc="left")
a1.legend(title="time since the rate change"); plt.tight_layout()

On the left the funnel gets deeper at the well and wider with time. On the right, against log distance, each curve is a straight line out to some radius and then bends down to zero. The bend is the edge of the funnel and it moves outward with time. Beyond it the pressure has not changed yet.

### A. Check against Theis

The model does not use the Theis solution. If both are correct, the pressure at the well wall against time must follow $\Delta P = \dfrac{W}{4\pi T} E_1\!\left(\dfrac{S_m r_\mathrm{well}^2}{4Tt}\right)$.

In [ ]:
def theis(r, t, T, S_m, W):
    '''Pressure change (Pa) at distance r (m) and time t (s) for a constant rate change W (kg/s) from t = 0.'''
    return W / (4 * np.pi * T) * exp1(S_m * r**2 / (4 * T * t))

t_s = np.logspace(0, np.log10(1000 * 3600), 60)                # 1 s to 1000 h
sol2 = solve_ivp(dPdt, (0, t_s[-1]), np.zeros(len(r)), t_eval=t_s, method="BDF", rtol=1e-6, atol=1.0)
p_well_model = sol2.y[0, :] / BAR
p_well_theis = theis(r[0], t_s, T_k37, Sm_k37, W_k37) / BAR

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(t_s / 60, p_well_model, "o", ms=5, mfc="none", label="finite-difference model, pressure at the well wall")
ax.semilogx(t_s / 60, p_well_theis, "-", color="#eb6834", label="Theis solution")
ax.set_xlabel("time since the rate change, min (log)"); ax.set_ylabel("pressure change at the well, bar"); ax.legend()
ax.set_title("The straight line against log time, from both", loc="left")
print("largest difference after 1 min: %.3f bar" % np.max(np.abs(p_well_model - p_well_theis)[t_s > 60]))

### B2.1 What T and S_m do

Run the model twice more: once with half the transmissivity, once with ten times the storativity. Enter the two changed values; the code runs the three cases, plots the pressure at the well against log time and fits the slope of the late part. Which change alters the slope of the straight line, and which only moves the line up or down?

In [ ]:
T_half = ___   # half the transmissivity, m s
Sm_x10 = ___   # ten times the storativity, kg/(m2 Pa)
cases = {"K-37": (T_k37, Sm_k37), "T / 2": (T_half, Sm_k37), "S_m x 10": (T_k37, Sm_x10)}

fig, ax = plt.subplots(figsize=(9, 5))
slopes = {}
for name, (T_, Sm_) in cases.items():
    r_, f_ = make_model(T_, Sm_, W_k37)
    s_ = solve_ivp(f_, (0, t_s[-1]), np.zeros(len(r_)), t_eval=t_s, method="BDF", rtol=1e-6, atol=1.0)
    pw = s_.y[0, :] / BAR
    ax.semilogx(t_s / 60, pw, label=name)
    late = t_s > 3600                                                    # after one hour
    slopes[name] = np.polyfit(np.log10(t_s[late] / 60), pw[late], 1)[0]  # bar per log cycle, from a line fit to the late part
ax.set_xlabel("time since the rate change, min (log)"); ax.set_ylabel("pressure change at the well, bar"); ax.legend()
for k, v in slopes.items(): print(f"{k:7s}: slope {v:.2f} bar per log cycle")

Halving $T$ doubles the slope. Ten times $S_m$ leaves the slope unchanged and moves the line down. The slope is the measurement of $T$; the height of the line is where $S_m$ hides, together with the well radius and the skin.

### Possible extensions

1. Put a sealed boundary at 100 m instead of 2000 m (`r_outer=100`) and make the outer face sealed, `w[-1] = 0`. What happens to the slope once the funnel reaches the edge? Compare with the boundaries slide.
2. Make `W` a function of time, 30 kg/s for six hours then zero, and reproduce the falloff curves from the lecture.
3. Replace the single `T` by an array that is ten times smaller inside r = 1 m (a damaged well wall) and see what skin does to the record.

## Part 3. Transmissivity and kh from the straight line

For a single well after the first minutes the Theis solution is a straight line against $\log_{10} t$ with slope

$$ m = \frac{2.303\,W}{4\pi T}, \qquad\text{so}\qquad T = \frac{2.303\,W}{4\pi\,|m|}, \qquad kh = T\,\nu. $$

In class you drew the line on paper. Here the computer fits it by least squares through the straight part of the record. You still have to decide which part of the record is straight.

### A. K-37 step 3 against log time

Step 3 is the rate decrease from 50 to 20 L/s at 20:18. The cell plots the pressure against the logarithm of the time since the rate change. Look for the straight part: not the first minutes (the well itself is still adjusting) and not the end (the line flattens).

In [ ]:
s3 = k37[k37.step == 3].copy()
s3["t_min"] = (s3.t_s - s3.t_s.min()) / 60 + 5 / 60      # minutes since the rate change (the first row is 5 s in)

def plot_semilog(t_min, p_barg, title):
    '''Pressure against log time since a rate change.'''
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.semilogx(t_min, p_barg, ".", ms=4)
    ax.set_xlabel("minutes since the rate change (log scale)"); ax.set_ylabel("pressure at the feed zone, bar-g")
    ax.set_title(title, loc="left")
    return ax

plot_semilog(s3.t_min, s3.p_barg, "K-37 step 3, 50 to 20 L/s");

### B3.1 Fit the line, compute T and kh

Enter the start and end of the straight part in minutes. The code fits the line between them. Then compute $T$ and $kh$ from the slope. Use the hot reservoir water for $\nu$, not the cold injected water: the pressure change travels through the reservoir fluid. The feed zone is at 80 bar-g and the Krafla reservoir boils at this depth, so the reservoir liquid is saturated water at that pressure. `P_feed_MPa` (8.1 MPa) was defined in Part 2 and `sat_liquid(P_MPa)` from Part 0 returns the boiling temperature, density and kinematic viscosity of saturated liquid water at that pressure.

In [ ]:
start_min = ___   # start of the straight part, minutes after the rate change
end_min = ___   # end of the straight part, minutes

fit_window = (s3.t_min >= start_min) & (s3.t_min <= end_min)
m_bar, icpt = np.polyfit(np.log10(s3.t_min[fit_window]), s3.p_barg[fit_window], 1)   # slope (bar per log cycle) and intercept

W = 30.0                                          # kg/s, the rate change 50 to 20 L/s of cold water
T_sat, rho_hot, nu_hot = ___   # saturated liquid water at the feed-zone pressure
T = ___   # transmissivity, m s (BAR converts the slope from bar to Pa)
kh = ___   # permeability-thickness, m3

print(f"reservoir liquid at {P_feed_MPa:.1f} MPa: boils at {T_sat:.0f} C, rho = {rho_hot:.0f} kg/m3, nu = {nu_hot:.2e} m2/s (cold water: {nu_water(20, 0.1):.2e})")
print(f"slope m = {abs(m_bar):.2f} bar per log cycle")
print(f"T  = {T:.2e} m s")
print(f"kh = {kh:.2e} m3 = {kh / DARCY_M:.1f} darcy-metres")

ax = plot_semilog(s3.t_min, s3.p_barg, "K-37 step 3 with the fitted line")
tt = np.array([start_min, end_min])
ax.semilogx(tt, m_bar * np.log10(tt) + icpt, color="#eb6834", lw=2.5, label=f"line through {start_min} to {end_min} min, slope {abs(m_bar):.2f} bar per cycle")
ax.legend();

### A. Why the fit window matters

Fit the same record over four different windows and see how much $kh$ moves. This is the uncertainty of the method, and it is larger than the scatter of the data.

In [ ]:
for lo, hi in [(1, 60), (4, 60), (4, 150), (10, 150)]:
    w_ = (s3.t_min >= lo) & (s3.t_min <= hi)
    m_ = np.polyfit(np.log10(s3.t_min[w_]), s3.p_barg[w_], 1)[0]
    kh_ = 2.303 * W / (4 * np.pi * abs(m_) * BAR) * nu_hot
    print(f"fit {lo:3d} to {hi:3d} min: slope {abs(m_):.2f} bar/cycle, kh = {kh_/DARCY_M:.1f} darcy-m")
print("Before 4 min the record is still affected by wellbore storage; after about 60 min the line flattens. Both reduce the fitted slope.")

### A. The first step of K-38 and NJ-15 against log time

Same plot for the first rate change of the other two wells: K-38 from 45 to 25 L/s at 02:59, NJ-15 from 28.9 to 18.9 L/s at 03:49 (times from the pump schedule in the file headers). Look at both and decide where the straight part starts and ends. The points are two minutes apart, so the first few minutes are not resolved.

In [ ]:
def step_data(df, t_change_h, t_next_h):
    '''Pressure of one step as minutes since the rate change, from a record with columns t_h and p_barg.'''
    d = df[(df.t_h > t_change_h) & (df.t_h < t_next_h)].copy()
    d["t_min"] = (d.t_h - t_change_h) * 60
    return d

k38_step1 = step_data(k38_rec, 2 + 59 / 60, 5 + 58 / 60)      # 02:59 to 05:58
nj15_step1 = step_data(nj15_rec, 3 + 49 / 60, 6 + 20 / 60)    # 03:49 to 06:20
plot_semilog(k38_step1.t_min, k38_step1.p_barg, "K-38 step 1, 45 to 25 L/s")
plot_semilog(nj15_step1.t_min, nj15_step1.p_barg, "NJ-15 step 1, 28.9 to 18.9 L/s");

### B3.2 T and kh for K-38 and NJ-15

Enter the fit window for each well in minutes. The code fits the line and gives you the slope `m_`; write the equation for `T_` and `kh_`. The rate changes are 20 kg/s (K-38) and 10 kg/s (NJ-15). Feed-zone conditions: K-38 about 300 C at 117 bar-g, NJ-15 about 280 C at 137 bar-g (temperature logs in Ntihabose, 2014); both are liquid water at these conditions, so `nu_water(T_C, P_MPa)` from Part 0 gives $\nu$.

In [ ]:
k38_start, k38_end = ___   # fit window for K-38, minutes after the rate change
nj15_start, nj15_end = ___   # fit window for NJ-15, minutes after the rate change

wells = {"K-38":  dict(data=k38_step1,  dW=20.0, T_C=300, P_MPa=11.8, start_min=k38_start,  end_min=k38_end),
         "NJ-15": dict(data=nj15_step1, dW=10.0, T_C=280, P_MPa=13.8, start_min=nj15_start, end_min=nj15_end)}

results = [dict(well="K-37", step="50 to 20 L/s", m_bar=abs(m_bar), T=T, kh_dm=kh / DARCY_M, II=II[3])]
for name, w in wells.items():
    d = w["data"]
    w_ = (d.t_min >= w["start_min"]) & (d.t_min <= w["end_min"])
    m_, c_ = np.polyfit(np.log10(d.t_min[w_]), d.p_barg[w_], 1)         # slope in bar per log cycle and intercept
    nu_ = nu_water(w["T_C"], w["P_MPa"])                                 # kinematic viscosity of the reservoir water, m2/s
    T_ = ___   # transmissivity, m s
    kh_ = ___   # permeability-thickness, m3
    results.append(dict(well=name, step="first rate change", m_bar=abs(m_), T=T_, kh_dm=kh_ / DARCY_M, II=steps_all[name].II.iloc[0]))
    ax = plot_semilog(d.t_min, d.p_barg, f"{name} step 1 with the fitted line")
    tt = np.array([w["start_min"], w["end_min"]])
    ax.semilogx(tt, m_ * np.log10(tt) + c_, color="#eb6834", lw=2.5, label=f"{w['start_min']} to {w['end_min']} min, slope {abs(m_):.2f} bar per cycle"); ax.legend()

results = pd.DataFrame(results)
display(results.assign(T=results["T"].map("{:.2e}".format)).round(2))

### A. Compare the index with kh

The steady radial flow equation gives $II = 2\pi T / \ln(r_e/r_\mathrm{well})$ for a well without skin. The cell computes this from the fitted $T$ with $\ln(r_e/r_\mathrm{well}) = 7$ and compares it with the measured index. The ratio of the two is a measure of the skin.

In [ ]:
results["II_from_kh"] = 2 * np.pi * results["T"] / 7 * BAR          # (kg/s) per bar, with ln(re/rwell) = 7
results["ratio"] = results["II"] / results["II_from_kh"]
display(results[["well", "kh_dm", "II_from_kh", "II", "ratio"]].round(2))
print("A ratio above 1 means the measured index is higher than the one computed from kh: negative skin, open fractures at the well wall.")

## Part 4. Interference: storativity from a second well

At the pumped well the height of the straight line depends on $S_m$, $r_\mathrm{well}$ and skin together, so $S_m$ cannot be read there. In an observation well the distance $r$ is known and there is no skin, so the height of the line gives $S_m$:

$$ S_m = \frac{2.25\,T\,t_0}{r^2}, \qquad S = S_m/\rho $$

where $t_0$ is the time at which the straight line, extended backwards, crosses zero pressure change.

Ohaaki (Broadlands), New Zealand, 1980. Well BR19 was opened and produced 64 kg/s. The water level in BR23, 350 m away, was recorded for 400 hours. The water at the top of BR23 is cold, so 1 m of water level is `bar_per_m_cold` from Part 0. The file gives the magnitude of the water level change; the level fell, because BR19 was producing.

In [ ]:
header("BR23_interference_1980.csv", 3)
br = load("BR23_interference_1980.csv")
br["dp_bar"] = br.dh_m * bar_per_m_cold          # water level drop to pressure drop
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(br.t_h, br.dh_m, ".", ms=4)
ax.set_xlabel("hours since BR19 was opened (log scale)"); ax.set_ylabel("water level drop in BR23, m")
ax.set_title("BR23: the straight part begins after about 100 h", loc="left");

### B4.1 Slope, T, zero crossing, S_m

Enter the start and end of the straight part in hours. The code fits the line, converts the slope to bar per log cycle, computes $T$, finds where the line crosses zero, and computes $S_m$ and $S$. $W = 64$ kg/s, $r = 350$ m, feed-zone temperature about 275 C (Grant and Bixley, 2011).

In [ ]:
start_h = ___   # start of the straight part, hours
end_h = ___   # end of the straight part, hours

W_br, r_br = 64.0, 350.0                                       # kg/s, m
w_ = (br.t_h >= start_h) & (br.t_h <= end_h)
m_m, c_m = np.polyfit(np.log10(br.t_h[w_]), br.dh_m[w_], 1)    # slope in m per log cycle and intercept
m_bar_br = m_m * bar_per_m_cold                                # slope in bar per log cycle
T_br = 2.303 * W_br / (4 * np.pi * m_bar_br * BAR)             # transmissivity, m s
t0_h = 10 ** (-c_m / m_m)                                      # hours where the line crosses zero: m log10(t0) + c = 0
T_feed_BR = 275                                                # C, feed-zone temperature
rho_br, nu_br = rho_water(T_feed_BR), nu_water(T_feed_BR)      # liquid water at the feed-zone temperature
Sm_br = 2.25 * T_br * t0_h * 3600 / r_br**2                    # mass storativity, kg/(m2 Pa)

print(f"slope {m_m:.2f} m per cycle = {m_bar_br:.3f} bar per cycle")
print(f"T = {T_br:.2e} m s,  kh = {T_br * nu_br / DARCY_M:.0f} darcy-metres")
print(f"line crosses zero at t0 = {t0_h:.0f} h,  S_m = {Sm_br:.1e} kg/(m2 Pa), so S = {Sm_br / rho_br:.1e} m/Pa")

fig, ax = plt.subplots(figsize=(9, 5))
tt = np.logspace(np.log10(t0_h) - 0.1, np.log10(end_h) + 0.1, 50)
ax.semilogx(br.t_h, br.dh_m, ".", ms=4, label="BR23 data")
ax.semilogx(tt, m_m * np.log10(tt) + c_m, color="#eb6834", lw=2, label=f"line through {start_h} to {end_h} h, slope {m_m:.2f} m per cycle")
ax.axhline(0, color="#888888", lw=0.8); ax.plot([t0_h], [0], "o", color="#eb6834", mfc="white", ms=8)
ax.set_ylim(-0.3, 2.5); ax.set_xlabel("hours since BR19 was opened (log scale)"); ax.set_ylabel("water level drop in BR23, m"); ax.legend();

Grant and Bixley (2011) report 80 darcy-metres and S = 5.7e-7 m/Pa for this test, from a line crossing zero at 11 h. The least-squares line through 100 to 400 h crosses zero near 30 h and gives about 1.6e-6 m/Pa. The zero crossing is an extrapolation over more than a decade in time and is uncertain by a factor of two to three; T from the slope is not.

### A. Fit the whole curve, not just the straight part

The early points lie above the straight line because at 350 m the straight part has not started yet. The full Theis curve describes these points too. `curve_fit` adjusts $T$ and $S_m$ until the curve matches all the points. We fit the logarithms of $T$ and $S_m$ so that the solver works with numbers of order one.

In [ ]:
def theis_fit(t_h, logT, logS):
    return theis(r_br, t_h * 3600, 10**logT, 10**logS, W_br) / BAR / bar_per_m_cold    # water level drop, m

p_opt, p_cov = curve_fit(theis_fit, br.t_h, br.dh_m, p0=[np.log10(T_br), np.log10(Sm_br)])
T_fit, S_fit = 10**p_opt
print(f"full-curve fit: T = {T_fit:.2e} m s (kh = {T_fit * nu_br / DARCY_M:.0f} darcy-m), S = {S_fit / rho_br:.1e} m/Pa")

fig, ax = plt.subplots(figsize=(9, 5))
tt = np.logspace(0, np.log10(500), 100)
ax.semilogx(br.t_h, br.dh_m, ".", ms=4, label="BR23 data")
ax.semilogx(tt, m_m * np.log10(tt) + c_m, "--", color="#52514e", label=f"straight line, {start_h} to {end_h} h")
ax.semilogx(tt, theis_fit(tt, *p_opt), color="#eb6834", lw=2, label="Theis curve fitted to all points")
ax.set_ylim(-0.2, 2.5); ax.set_xlabel("hours since BR19 was opened (log scale)"); ax.set_ylabel("water level drop in BR23, m"); ax.legend();

### A. The same thing from the Part 2 model

The ring model from Part 2 can also predict BR23. Put the fitted Ohaaki $T$ and $S_m$ in, run it for 400 hours, and read the pressure at the ring nearest to 350 m. This checks that the model, the Theis solution and the data agree.

In [ ]:
r_o, f_o = make_model(T_fit, S_fit, W_br, r_outer=20000.0, N=300)
i350 = np.argmin(np.abs(r_o - 350))
s_o = solve_ivp(f_o, (0, 400 * 3600), np.zeros(len(r_o)), t_eval=br.t_h.values * 3600, method="BDF", rtol=1e-6, atol=1.0)
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(br.t_h, br.dh_m, ".", ms=4, label="BR23 data")
ax.semilogx(br.t_h, s_o.y[i350, :] / BAR / bar_per_m_cold, color="#1baf7a", lw=2, label=f"ring model at r = {r_o[i350]:.0f} m")
ax.set_xlabel("hours since BR19 was opened (log scale)"); ax.set_ylabel("water level drop in BR23, m"); ax.legend();

### Possible extensions

1. How late would the straight line arrive at 1 km? Use the model or Theis with the Ohaaki numbers.
2. The storativity found here is about 30 times the liquid value computed for K-37 in Part 2. Rerun the full-curve fit with $S_m$ fixed at that liquid value and look at how badly the early points fit. A storativity far above the liquid value is one indication that the reservoir is boiling.

## Part 5. A discharge test and the James lip-pressure method

For a flowing well the mass flow is not measured directly. The well discharges into a silencer. Two things are measured: the lip pressure at the end of the discharge pipe, and the water flow over a weir after separation at atmospheric pressure. The James formula then gives the enthalpy and the total flow (Grant and Bixley 8.16 to 8.18):

$$ Y = \frac{W_w}{A\,P_\mathrm{lip}^{0.96}}, \qquad H = \frac{2675 + 3329\,Y}{1 + 28.3\,Y}, \qquad W = \frac{2258\,W_w}{2675 - H} $$

with $W_w$ the weir flow in kg/s, $A$ the lip pipe area in cm², $P_\mathrm{lip}$ in bar absolute, $H$ in kJ/kg. Grant and Bixley Table 8.3 gives twelve measurements on well BR20 at Ohaaki. The cell computes $W$ and $H$ for each and plots the output curve, total flow against wellhead pressure.

In [ ]:
header("BR20_output_test.csv", 3)
br20 = load("BR20_output_test.csv")
A_cm2 = np.pi * (br20.lip_diameter_mm / 10 / 2)**2            # lip pipe area, cm2
Ww = br20.water_flow_t_per_h * 1000 / 3600                    # weir flow, kg/s
P_lip_abs = br20.lip_pressure_barg + 1.0                      # bar absolute
Y = Ww / (A_cm2 * P_lip_abs**0.96)
H = (2675 + 3329 * Y) / (1 + 28.3 * Y)                         # kJ/kg
Wtot = 2258 * Ww / (2675 - H)                                  # kg/s
br20["H_kJkg"] = H; br20["W_kgs"] = Wtot
display(br20.round(1))

slope_whp, icpt_whp = np.polyfit(br20.WHP_barg, br20.W_kgs, 1)   # kg/s per bar of wellhead pressure
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(br20.WHP_barg, br20.W_kgs, "o", label="James method, twelve measurements")
pp = np.array([br20.WHP_barg.min(), br20.WHP_barg.max()])
ax.plot(pp, slope_whp * pp + icpt_whp, color="#eb6834", label=f"line, {abs(slope_whp):.1f} kg/s per bar of wellhead pressure")
ax.set_xlabel("wellhead pressure, bar-g"); ax.set_ylabel("total mass flow, kg/s"); ax.set_title("BR20 output curve", loc="left"); ax.legend()
T_res_BR20 = IAPWS97(P=10.0, h=H.median()).T - 273.15        # C, liquid water with this enthalpy (pressure has little effect)
print("Enthalpy near %.0f kJ/kg: liquid water at about %.0f C in the reservoir, boiling in the well on the way up." % (H.median(), T_res_BR20))

The slope of this line is flow per bar of wellhead pressure. It is not the productivity index. The productivity index is flow per bar of pressure change at the feed zone, and between the feed zone and the wellhead the fluid boils, so the wellhead pressure and the feed-zone pressure are not related by a constant. To get PI for a flowing well you need a gauge at the feed zone during the test, as for RN-30 in the lecture. The output curve is what the plant operator needs; PI is what the reservoir engineer needs.

## Assignment 3: hand in this notebook

Hand in this notebook (File > Download > Download .ipynb) with every cell run and every plot visible, together with short answers to the questions below, written in the last cell. Deadline and submission are on Canvas.

**Numbers to report** (from your completed cells)

1. The injectivity index of K-38 and NJ-15 for every rate change (B1.2), with units.
2. The fit window you chose for the first step of K-38 and of NJ-15, the slope in bar per log cycle, $T$ in m s and $kh$ in darcy-metres (B3.2). Say in one sentence why you chose each window.
3. For BR23, the fit window, $T$, $kh$, $t_0$ and $S_m$ (B4.1).

**Plots to include**

The step-test records of K-38 and NJ-15 (Part 1), the semilog plots with your fitted lines for both wells (B3.2), and the BR23 plot with the line and the zero crossing (B4.1).

**Questions** (three or four sentences each)

1. The index and $kh$ did not agree for K-37. Which one is a property of the rock, and what does the other one contain in addition?
2. K-37 and K-38 are neighbouring wells in Krafla. Compare their indices and their $kh$ values. Which of the two numbers differs more between the wells, and what could be the reason?
3. What did the ring model in Part 2 let you do that the Theis formula alone did not?
4. Which number from this notebook would you trust most if you had to decide whether to connect K-37 to the power plant, and why?

*Your answers here.*